# Corrigendum: Exp. 1 JSON scoring

All tables below are regenerated from `llm_predictions/exp1_predictions.csv`. `outcome_stored` reproduces the paper; `outcome_corrected` applies the corrected JSON assertion described in `corrigendum.md`.

In [ ]:
import os
from pathlib import Path
import pandas as pd
root = Path(os.environ.get('DIALEXPOL_DATA_DIR', Path.cwd().parent))
p = root / 'llm_predictions' / 'exp1_predictions.csv'
df = pd.read_csv(p, dtype=str, keep_default_na=False)
required = {'run','provider','template','outcome_stored','outcome_corrected','response_status'}
missing = required - set(df.columns)
assert not missing, f'missing prediction columns: {sorted(missing)}'
print(f'rows: {len(df):,}')


In [ ]:
print('Stored versus corrected outcomes')
for col in ('outcome_stored','outcome_corrected'):
    print(col, df[col].value_counts().to_dict())
json = df[df.template.str.endswith('_json')]
print('JSON accuracy stored/corrected:', (json.outcome_stored=='correct').mean(), (json.outcome_corrected=='correct').mean())
print('movement counts: stored-correct → corrected-wrong =', ((json.outcome_stored=='correct') & (json.outcome_corrected=='wrong')).sum())
print('stored-wrong → corrected-correct =', ((json.outcome_stored=='wrong') & (json.outcome_corrected=='correct')).sum())
print('total changed =', (json.outcome_stored != json.outcome_corrected).sum())


In [ ]:
print('Provider × template before/after')
tab = (json.assign(stored=json.outcome_stored.eq('correct'), corrected=json.outcome_corrected.eq('correct'))
        .groupby(['provider','template'])[['stored','corrected']].mean().mul(100).round(2))
display(tab)
print('format failures and no-response by run')
print(df.groupby('run').apply(lambda g: pd.Series({'stored_format_failure': (g.outcome_stored=='format_failure').sum(), 'corrected_format_failure': (g.outcome_corrected=='format_failure').sum(), 'no_response': (g.response_status=='no_response').sum()}), include_groups=False))


In [ ]:
unaffected = ~df.run.isin(['phase_a_json_restricted'])
assert (df.loc[unaffected, 'outcome_stored'] == df.loc[unaffected, 'outcome_corrected']).all()
print('unaffected-run equality: PASS')
